## 02 embedding 2.0 concat 
### 1 embedding 

In [1]:
import torch
import h5py
from tqdm import tqdm
import  esm

In [2]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path, marker: str = "src") -> Path:
    """
    Walk upward from the notebook's directory until we find
    a folder containing 'src'. That folder is the project root.
    """
    current = start.resolve()
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise RuntimeError("Project root not found. Make sure 'src/' exists.")

# Detect project root automatically
PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Detected project root:", PROJECT_ROOT)


Detected project root: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook


In [3]:
%cd "$PROJECT_ROOT"
!ls

/home/yi-jin/Documents/CATA6-protein-prediction-4staroverook
checkpoints  data     logs	 pipelines  src		 tmpDir
configs      LICENSE  notebooks  README.md  submissions


/home/yi-jin/anaconda3/envs/cafa6/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import os

data_raw_path = PROJECT_ROOT / "data" / "raw"
print("data/raw exists:", os.path.exists(data_raw_path))
print(os.listdir(data_raw_path) if os.path.exists(data_raw_path) else "MISSING")

train_fasta_path = PROJECT_ROOT / "data" / "raw" / "Train"
print(os.path.exists(train_fasta_path))
print(os.listdir(train_fasta_path) if os.path.exists(train_fasta_path) else "MISSING")


data/raw exists: True
['sample_submission.tsv', 'Test', 'Train', '.gitkeep', 'IA.tsv', '.DS_Store']
True
['go-basic.obo', 'train_terms.tsv', '.ipynb_checkpoints', 'train_taxonomy.tsv', 'train_sequences.fasta']


#### 1.1 load fasta, ID splits 

In [5]:

from src.preprocessing.prepare_embedding import (
    load_train_val_test_sequences,
    load_length_bins
)

train_seqs, val_seqs, test_seqs = load_train_val_test_sequences(PROJECT_ROOT)
len(train_seqs), len(val_seqs), len(test_seqs)




(70093, 12311, 224309)

In [6]:
# confirm binning 
train_bins, val_bins, test_bins = load_length_bins(train_seqs, val_seqs, test_seqs)

for name, bin_dict in [("Train", train_bins), ("Val", val_bins), ("Test", test_bins)]:
    print(f"=== {name} ===")
    for k, v in bin_dict.items():
        print(k, len(v))


=== Train ===
short_<=1022 63557
mid_1023_2048 5455
long_2049_5000 1017
ultra_>5000 64
=== Val ===
short_<=1022 11192
mid_1023_2048 927
long_2049_5000 173
ultra_>5000 19
=== Test ===
short_<=1022 210921
mid_1023_2048 11172
long_2049_5000 2066
ultra_>5000 150


#### 1.2 Load ESM2 Model 

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.to(device)
model.eval()

batch_converter = alphabet.get_batch_converter()


Device: cuda


In [8]:
#### run a minitest to confirm the setting
train_seqs, val_seqs, test_seqs = load_train_val_test_sequences(PROJECT_ROOT)

# mini test: first 10 sequences
mini = train_seqs[:10]
len(mini), mini[:2]


(10,
 [('Q8R4S6',
   'MSSGANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTELDRLASLLPFPQDVINKLDKLSVLRLSVSYLRAKSFFDVALKSTPADRNGGQDQCRAQIRDWQDLQEGEFLLQALNGFVLVVTADALVFYASSTIQDYLGFQQSDVIHQSVYELIHTEDRAEFQRQLHWALNPDSAQGVDEAHGPPQAAVYYTPDQLPPENASFMERCFRCRLRCLLDNSSGFLAMNFQGRLKYLHGQNKKGKDGALLPPQLALFAIATPLQPPSILEIRTKNFIFRTKHKLDFTPIGCDAKGQLILGYTEVELCTRGSGYQFIHAADMLHCAESHIRMIKTGESGMTVFRLLAKHSRWRWVQSNARLIYRNGRPDYIIVTQRPLTDEEGREHLQKRSTSLPFMFATGEAVLYEISSPFSPIMDPLPIRTKSNTSRKDWAPQSTPSKDSFHPSSLMSALIQQDESIYLCPPSSPAPLDSHFLMGSVSKCGSWQDSFAAAGSEAALKHEQIGHAQDVNLALSGGPSELFPDNKNNDLYNIMRNLGIDFEDIRSMQNEEFFRTDSTAAGEVDFKDIDITDEILTYMQDSLNNSTLMNSACQQQPVTQHLSCMLQERLQLEQQQQLQQPPPQALEPQQQLCQMVCPQQDLGPKHTQINGTFASWNPTPPVSFNCPQQELKHYQLFSSLQGTAQEFPYKPEVDSVPYTQNFAPCNQPLLPEHSKSVQLDFPGRDFEPSLHPTTSNLDFVSCLQVPENQSHGINSQSAMVSPQAYYAGAMSMYQCQPGPQRTPVDQTQYSSEIPGSQAFLSKVQSRGIFNETYSSDLSSIGHAAQTTGHLHHLAEARPLPDITPGGFL'),
  ('O74382',
   'MVNAAIVVSSNRPNVLEQISSLFRDGKTRSLGEQWTLVSGQLKGTFEDAKDACNRISATENVDCNCLSEATFSTKKKLVVFDMDSTLIQQECIDELAAEAGIQKEVATI

In [8]:
from src.preprocessing.esm_concat import embed_concat_batch, embed_bin_concat
test_batch = mini[:4]
emb_dict = embed_concat_batch(model, batch_converter, device, test_batch)

list(emb_dict.keys()), emb_dict[next(iter(emb_dict))].shape


(['Q9I6J2', 'Q9D311', 'Q6ENH4', 'Q05594'], (2560,))

#### 1.3 Embedding 

In [10]:
from src.preprocessing.esm_concat import run_concat_embedding
from pathlib import Path

# Detect project root automatically

BASE = PROJECT_ROOT

# Choose a clear output file name
OUTPUT = f"{BASE}/data/embeddings/esm2_650M_trainval_concat_2560.h5"

print("BASE:", BASE)
print("OUTPUT:", OUTPUT)

run_concat_embedding(BASE, OUTPUT, mode= "trainval")


BASE: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook
OUTPUT: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook/data/embeddings/esm2_650M_trainval_concat_2560.h5
Running on device: cuda
Loading ESM2-650M model...
Loading sequences...
Train: 70093  Val: 12311  Test: 224309

 STARTING TRAIN EMBEDDING 

Processing bin: short_<=1022
  Remaining in this bin: 0


0it [00:00, ?it/s]


Processing bin: mid_1023_2048
  Remaining in this bin: 0


0it [00:00, ?it/s]


Processing bin: long_2049_5000
  Remaining in this bin: 0


...

Validation bin: long_2049_5000
  Remaining in this bin: 0


0it [00:00, ?it/s]


Validation bin: ultra_>5000
  Remaining in this bin: 0


0it [00:00, ?it/s]



=== CONCAT EMBEDDING COMPLETE ===
Saved to: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook/data/embeddings/esm2_650M_trainval_concat_2560.h5


In [ ]:
# after completion, check out put. 

In [ ]:
# run test seperately 

In [9]:
from src.preprocessing.esm_concat import run_concat_embedding
from pathlib import Path

OUTPUT_TEST = f"{PROJECT_ROOT}/data/embeddings/esm2_650M_test_concat_2560.h5"
run_concat_embedding(PROJECT_ROOT, OUTPUT_TEST, mode="test")


Running on device: cuda
Loading ESM2-650M model...
Loading sequences...
Train: 70093 | Val: 12311 | Test: 224309

 STARTING TEST EMBEDDING 

[Test] Bin: short_<=1022
  Remaining in this bin: 210921


100%|███████████████████████████████████████████████████████████████████████| 26366/26366 [2:30:35<00:00,  2.92it/s]


[Test] Bin: mid_1023_2048
  Remaining in this bin: 11172


100%|███████████████████████████████████████████████████████████████████████████| 1397/1397 [11:44<00:00,  1.98it/s]


[Test] Bin: long_2049_5000
  Remaining in this bin: 2066


100%|█████████████████████████████████████████████████████████████████████████████| 259/259 [02:11<00:00,  1.97it/s]


[Test] Bin: ultra_>5000
  Remaining in this bin: 150


100%|███████████████████████████████████████████████████████████████████████████████| 19/19 [00:09<00:00,  1.99it/s]


=== CONCAT EMBEDDING COMPLETE ===
Saved to: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook/data/embeddings/esm2_650M_test_concat_2560.h5


### 2. Embedding function test module 

1)FASTA → embedding pipeline → emb_dict
2)GO annotation CSV → label loader → label_dict
3) train_ids CSV → splitting → protein_ids list

1,2,3 construct
ProteinDataset(emb_dict, label_dict, train_ids)


GO prediction pipleline 

Load embeddings (from HDF5)
Load GO labels (protein_id → multi-hot label vector)
Construct ProteinDataset(embeddings, labels, IDs)
Wrap in DataLoader
Train model

1. Data pipeline 
Inputs:
   - embeddings.h5 file (depends on which embedding methods)
   - GO annotation matrix
   - train/val protein IDs
   - multi-label binarieid GO targets 

Modules needed: 
src/dataloader/embedding_loader.py (done)
    - load HDF5 embeddings
    - store in dict {protein_id: embedding_vector}
    - sed by dataset class 
    
src/dataloader/dataset.py 
    - it gives embedding(tensor), labels(tensor) 
    
src/dataloader/splitter.py 
    - loads train_id_40.csv and val_id_40.csv 
    - returns list of protein IDs for each split

2. MODEL PIPELINE
Architecture:
    input: 2560-d embedding
    hidden: layers: 1024/512
    dropout: 0.3?
    Activation: Relu?
    Output layer: num_GO_terms
    Loss: BCElogits
    optimizerL: AdamW
    Scheduler:
3. Train/Eval PIPELINE
Trainer Class:
    Train one epoch
    Validate one epoch
    compute Fmax, AUROC, PR-AUC
    save checkpoints

Metrics: 
    BCE loss
    Micro F1/ Macro F1/ Fmax(CAFA metric)/ Precision-Recall 



#### 2.1 test embedding_loader

In [6]:
from src.dataloader.embedding_loader import load_embeddings_h5
import os

# Path to your real trainval embedding file
h5_path = "data/embeddings/esm2_650M_trainval_concat_2560.h5"

print("File exists:", os.path.exists(h5_path))

# Load with metadata info, define emb_dict
emb_dict, info = load_embeddings_h5(h5_path, return_info=True)

print("\n=== EMBEDDING SUMMARY ===")
print("Number of proteins:", info["n_proteins"])
print("Embedding level:", info["embedding_level"])
print("Embedding dimension:", info["dimensionality"])
print("Example protein:", info["example_id"])
print("Example embedding shape:", emb_dict[info["example_id"]].shape)

# Print first values for sanity
print("\nFirst 10 values of example embedding:")
print(emb_dict[info["example_id"]][:10])


File exists: True

=== EMBEDDING SUMMARY ===
Number of proteins: 82924
Embedding level: protein
Embedding dimension: 2560
Example protein: A0A023FBW4
Example embedding shape: torch.Size([2560])

First 10 values of example embedding:
tensor([ 0.0704, -0.0008,  0.0192,  0.0429,  0.0446, -0.1005,  0.0631,  0.0462,
        -0.0215,  0.0965])


#### 2.2 test GO loader

In [7]:
from src.dataloader.go_label_loader import (
    load_go_terms, build_go_vocabulary, build_label_dictionary
)

# build GO vocab + labels. 
df = load_go_terms("data/raw/Train/train_terms.tsv")
go2idx, idx2go = build_go_vocabulary(df)

print("Number of GO terms:", len(go2idx))
print("First 5 GO terms:", idx2go[:5])

pid = df["entry_id"].iloc[0]
print("Example protein:", pid)

label_dict = build_label_dictionary(df, go2idx)
print("Label vector shape:", label_dict[pid].shape)
print("Number of positive labels:", int(label_dict[pid].sum()))


Number of GO terms: 26125
First 5 GO terms: ['GO:0000001', 'GO:0000002', 'GO:0000006', 'GO:0000007', 'GO:0000009']
Example protein: Q5W0B1
Label vector shape: torch.Size([26125])
Number of positive labels: 7


#### 2.3 Load train/val protein ID lists

In [15]:
import pandas as pd
def load_id_list(path):
    df = pd.read_csv(path)
    return set(df.iloc[:,0].astype(str))

train_ids = load_id_list("data/processed/train_id_40.csv")
val_ids = load_id_list("data/processed/val_id_40.csv")

print("Train IDs:" , len(train_ids))
print("Val Ids:", len(val_ids)) 



Train IDs: 70093
Val Ids: 12311


#### 2.4 Test ProteinDataset

In [17]:
import torch
from src.dataloader.dataset import ProteinDataset

train_ds = ProteinDataset(emb_dict, label_dict, train_ids)
val_ds   = ProteinDataset(emb_dict, label_dict, val_ids)

print("Train dataset size:", len(train_ds))
print("Val dataset size:", len(val_ds))

Train dataset size: 70093
Val dataset size: 12311


#### 2.5 Test get_items

In [18]:
x, y = train_ds[0]

print("Embedding shape:", x.shape)
print("Label shape:", y.shape)
print("Number of positive GO terms:", int(y.sum()))


Embedding shape: torch.Size([2560])
Label shape: torch.Size([26125])
Number of positive GO terms: 9


#### 2.6 Test DataLoader Batching 

In [19]:
from torch.utils.data import DataLoader

loader = DataLoader(train_ds, batch_size=8, shuffle=True)

batch_x, batch_y = next(iter(loader))
print("Batch X:", batch_x.shape)
print("Batch Y:", batch_y.shape)


Batch X: torch.Size([8, 2560])
Batch Y: torch.Size([8, 26125])


### 3. Trainer

#### 3.1 mlp classifier testing

In [9]:
from src.models.mlp import MLPClassifier
import torch

model = MLPClassifier(input_dim=2560, output_dim=5000)
x = torch.randn(8, 2560)
y = model(x)

print(y.shape)


torch.Size([8, 5000])


#### 3.2 Trainer test 

In [10]:
from src.models.mlp import MLPClassifier
from src.training.trainer import Trainer
import torch
from torch.utils.data import DataLoader, TensorDataset

# Dummy data
x = torch.randn(64, 2560)
y = torch.randint(0, 2, (64, 100)).float()

loader = DataLoader(TensorDataset(x, y), batch_size=8)

model = MLPClassifier(2560, 100)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

trainer = Trainer(
    model=model,
    device=torch.device("cpu"),
    criterion=criterion,
    optimizer=optimizer,
)

loss = trainer.train_one_epoch(loader)
print("Train loss:", loss)


Train loss: 0.6940086632966995


### 3.3 Train pipleline 

In [11]:
from pipelines.train_mlp import main
EPOCHS = 1
BATCH_SIZE = 32

main()


Using device: cuda

Loading embeddings...
Loaded 82924 embeddings (dim=2560)

Loading GO terms...
GO terms: 26125
Proteins with labels: 82404
Saved GO vocabulary → checkpoints/go_vocab.json

Building datasets...
Train dataset size: 70093
Val dataset size:   12311

Initializing model...

Starting training...



Epoch 01 | Train Loss: 0.0049 | Val Loss: 0.0018 | Fmax: 0.1701 @ t=0.06
  New best Fmax — saving checkpoint


Epoch 02 | Train Loss: 0.0019 | Val Loss: 0.0018 | Fmax: 0.1866 @ t=0.08
  New best Fmax — saving checkpoint


Epoch 03 | Train Loss: 0.0018 | Val Loss: 0.0017 | Fmax: 0.2091 @ t=0.12
  New best Fmax — saving checkpoint


Epoch 04 | Train Loss: 0.0017 | Val Loss: 0.0017 | Fmax: 0.2161 @ t=0.08
  New best Fmax — saving checkpoint


Epoch 05 | Train Loss: 0.0016 | Val Loss: 0.0016 | Fmax: 0.2247 @ t=0.12
  New best Fmax — saving checkpoint

Training complete.
Best Fmax: 0.2247
Checkpoint saved to: checkpoints/best_mlp.pt


#### 3.4 Test Inference 

In [9]:
import os
import sys

PROJECT_ROOT = os.path.abspath(".")
sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)
print(sys.path[:3])
!ls

/home/yi-jin/Documents/CATA6-protein-prediction-4staroverook
['/home/yi-jin/Documents/CATA6-protein-prediction-4staroverook', '/home/yi-jin/Documents/CATA6-protein-prediction-4staroverook', '/home/yi-jin/anaconda3/envs/cafa6/lib/python310.zip']
checkpoints  data     logs	 pipelines  src		 tmpDir
configs      LICENSE  notebooks  README.md  submissions


In [6]:
import sys
print(sys.executable)
import goatools
print(goatools.__version__)


/home/yi-jin/anaconda3/envs/cafa6/bin/python3.10
1.5.2


In [15]:
!python -m pipelines.predict_test

Error processing line 1 of /home/yi-jin/anaconda3/envs/cafa6/lib/python3.10/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "/home/yi-jin/anaconda3/envs/cafa6/lib/python3.10/site.py", line 195, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Using device: cuda

Loading test embeddings...
Loaded 224309 test proteins

Loading checkpoint...
Threshold: 0.120
Output dim: 26125
Loaded idx2go with 26125 terms. Example: ['GO:0000001', 'GO:0000002', 'GO:0000006', 'GO:0000007', 'GO:0000009']
Loading GO ontology...
data/raw/Train/go-basic.obo: fmt(1.2) rel(2025-06-01) 43,448 Terms
GO terms with ontology info: 43448
Ontology split — MF: 6616, BP: 16858, CC: 2651

Running ontology-aware inference...
Inference: 100%|███████████████████████████| 7010/7010 [00:59<00:00, 118.60it/s]

Inference complete.
Submission written to: submissions/submission.tsv
Total

In [16]:
import pandas as pd

df = pd.read_csv("submissions/submission.tsv", sep="\t", header=None)
df.columns = ["protein", "go", "score"]

df.groupby("protein").size().describe()


count    224309.000000
mean         52.681560
std          41.627516
min           5.000000
25%          27.000000
50%          42.000000
75%          63.000000
max         373.000000
dtype: float64

In [17]:
df.head()

,protein,go,score
0,A0A017SE81,GO:0005515,0.0895
1,A0A017SE81,GO:0042802,0.0441
2,A0A017SE81,GO:0042803,0.0436
3,A0A017SE81,GO:0016491,0.0276
4,A0A017SE81,GO:0005886,0.1944


In [18]:
df.tail()

,protein,go,score
11816943,X6R8R1,GO:0001938,0.0104
11816944,X6R8R1,GO:0051865,0.0103
11816945,X6R8R1,GO:0010628,0.0103
11816946,X6R8R1,GO:0030036,0.0103
11816947,X6R8R1,GO:0010977,0.0101
